In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import LSTM, Dense

from sklearn.model_selection import train_test_split

## Load Dataset

In [ ]:
df = pd.read_csv(
    "../../Datasets/Indonesian Sentiment Twitter Dataset Labeled.csv",
    sep="\t"
)
df.head()

### Lihat Informasi Dataset

In [ ]:
df.info()

In [ ]:
df["sentimen"].value_counts()

### Ubah Label

In [ ]:
df["sentimen"] = df["sentimen"].replace({
    -1: 0,
     0: 1,
     1: 2
})

df["sentimen"].value_counts()

## Train and Split

In [ ]:
import re

def clean_text(text):
    text = text.lower()

    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)

    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)

    return text

df["Tweet"] = df["Tweet"].astype(str).apply(clean_text)

X = df["Tweet"]
y = df["sentimen"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## TextVectorization

In [ ]:
max_words = 5000
max_len = 50

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=max_words,
    output_mode="int",
    output_sequence_length=max_len
)

vectorizer.adapt(X_train)

## Lihat Hasil Tokenisasi

In [ ]:
print("\nKalimat Asli:")
print(X_train.iloc[0])

print("\nHasil Vectorization:")
print(vectorizer([X_train.iloc[0]]).numpy())

## Membuat Model RNN

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(1,), dtype=tf.string),
    vectorizer,
    tf.keras.layers.Embedding(max_words, 64),
    tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(64)
    ),
    tf.keras.layers.Dense(3, activation="softmax")
])


## Compile

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## Training

In [ ]:
X_train = X_train.to_numpy(dtype=object)
X_test = X_test.to_numpy(dtype=object)

y_train = y_train.to_numpy(dtype=np.int32)
y_test = y_test.to_numpy(dtype=np.int32)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[early_stop]
)

## Evaluasi

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.title("Training Loss vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

plt.title("Training Accuracy vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

pred = model.predict(X_test)

pred = np.argmax(pred, axis=1)

print(confusion_matrix(y_test, pred))

## Predict

In [ ]:
label = {
    0: "Negatif",
    1: "Netral",
    2: "Positif"
}

kalimat = np.array(
    ["hei, hari ini ulang tahunku loh, kasih aku hadiah dong"],
    dtype=object
)

pred = model.predict(kalimat)

kelas = np.argmax(pred)

print("\nKalimat :", kalimat[0])
print("Probabilitas :", pred)
print("Prediksi :", label[kelas])